In [1]:
import pandas as pd
from pathlib import Path

In [3]:
datasets_dir = Path('datasets')

In [ ]:


excel_files = [
    'aihw-can-122-CDiA-2023-Book-1a-Cancer-incidence-age-standardised-rates-5-year-age-groups.xlsx',
    'aihw-can-122-CDiA-2023-Book-2a-Cancer-mortality-and-age-standardised-rates-by-age-5-year-groups.xlsx',
    'aihw-can-122-CDiA-2023-Book-7-Cancer-incidence-by-state-and-territory.xlsx',
]

for filename in excel_files:
    xlsx_path = datasets_dir / filename
    xf = pd.ExcelFile(xlsx_path)
    sheet_name = xf.sheet_names[1]  # second worksheet (index 1)
    df = pd.read_excel(xlsx_path, sheet_name=1)
    csv_path = datasets_dir / (Path(filename).stem + '.csv')
    df.to_csv(csv_path, index=False)
    print(f'{filename}')
    print(f'  Sheet: "{sheet_name}" | Shape: {df.shape}')
    print(f'  Saved -> {csv_path}\n')

aihw-can-122-CDiA-2023-Book-1a-Cancer-incidence-age-standardised-rates-5-year-age-groups.xlsx
  Sheet: "Table S1a.1" | Shape: (227405, 13)
  Saved -> datasets/aihw-can-122-CDiA-2023-Book-1a-Cancer-incidence-age-standardised-rates-5-year-age-groups.csv



aihw-can-122-CDiA-2023-Book-2a-Cancer-mortality-and-age-standardised-rates-by-age-5-year-groups.xlsx
  Sheet: "Table S2a.1" | Shape: (261203, 13)
  Saved -> datasets/aihw-can-122-CDiA-2023-Book-2a-Cancer-mortality-and-age-standardised-rates-by-age-5-year-groups.csv



aihw-can-122-CDiA-2023-Book-7-Cancer-incidence-by-state-and-territory.xlsx
  Sheet: "Table S7.1" | Shape: (69466, 10)
  Saved -> datasets/aihw-can-122-CDiA-2023-Book-7-Cancer-incidence-by-state-and-territory.csv



## Load CSVs

In [2]:
NA_VALUES = ['. .', 'n.p.', 'N/A', 'na', '']

# Book 1a — Cancer incidence by age group
df_incidence = pd.read_csv(
    datasets_dir / 'aihw-can-122-CDiA-2023-Book-1a-Cancer-incidence-age-standardised-rates-5-year-age-groups.csv',
    skiprows=5,
    na_values=NA_VALUES
)
df_incidence.columns = df_incidence.columns.str.strip()
# Drop fully empty trailing columns
df_incidence = df_incidence.dropna(axis=1, how='all')

print('df_incidence shape:', df_incidence.shape)
df_incidence.head()

NameError: name 'datasets_dir' is not defined

In [ ]:
# Book 2a — Cancer mortality by age group
df_mortality = pd.read_csv(
    datasets_dir / 'aihw-can-122-CDiA-2023-Book-2a-Cancer-mortality-and-age-standardised-rates-by-age-5-year-groups.csv',
    skiprows=5,
    na_values=NA_VALUES
)
df_mortality.columns = df_mortality.columns.str.strip()
df_mortality = df_mortality.dropna(axis=1, how='all')

print('df_mortality shape:', df_mortality.shape)
df_mortality.head()

In [ ]:
# Book 7 — Cancer incidence by state and territory
df_state = pd.read_csv(
    datasets_dir / 'aihw-can-122-CDiA-2023-Book-7-Cancer-incidence-by-state-and-territory.csv',
    skiprows=5,
    na_values=NA_VALUES
)
df_state.columns = df_state.columns.str.strip()
df_state = df_state.dropna(axis=1, how='all')

print('df_state shape:', df_state.shape)
df_state.head()

## Inspect & Clean

In [ ]:
for name, df in [('incidence', df_incidence), ('mortality', df_mortality), ('state', df_state)]:
    print(f'=== df_{name} ===')
    df.info()
    print(f'Null counts:\n{df.isnull().sum()}\n')

In [ ]:
# Standardise column names (lowercase, underscores)
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.lower()
        .str.replace(r'\s+', '_', regex=True)
        .str.replace(r'[^\w]', '_', regex=True)
        .str.strip('_')
    )
    return df

df_incidence = clean_columns(df_incidence)
df_mortality = clean_columns(df_mortality)
df_state     = clean_columns(df_state)

# Convert Count to numeric (coerce any stray strings to NaN)
for df in [df_incidence, df_mortality, df_state]:
    df['count'] = pd.to_numeric(df['count'], errors='coerce')
    df['year']  = pd.to_numeric(df['year'],  errors='coerce')

print('Columns — incidence:', df_incidence.columns.tolist())
print('Columns — mortality:', df_mortality.columns.tolist())
print('Columns — state:',    df_state.columns.tolist())